In [44]:


%pip install groq --quiet

print("✅ Groq SDK installed.")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Groq SDK installed.


In [45]:


import os, re, json, time, warnings
warnings.filterwarnings('ignore')

import requests
import pandas as pd
from tqdm.notebook import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Groq client — drop-in OpenAI-compatible interface
from groq import Groq, RateLimitError, APIStatusError

print("✅ All imports successful (including Groq).")

✅ All imports successful (including Groq).


In [ ]:

# ── Entity & Paths (unchanged from Phase 2) ───────────────────
entity_name     = "Tesla"
PHASE1_CSV_PATH = f"news_{entity_name.replace(' ', '_')}_phase1.csv"
OUTPUT_CSV_PATH = f"adverse_detection_{entity_name.replace(' ', '_')}_phase2.csv"

# ── Groq API Config ───────────────────────────────────────────

GROQ_API_KEY = ""  # YOUR API KEY 

# Model: llama-3.3-70b-versatile
# Why this model:
#   - Best reasoning quality on Groq's free tier
#   - Reliable JSON output with instruction-following
#   - 128K context window — handles long article summaries easily
#   - Free tier: 1000 requests/day, 6000 tokens/min
#
# Fallback (if quota exhausted): "llama-3.1-8b-instant"
#   Faster, lower quality, but same JSON reliability.
GROQ_MODEL = "llama-3.3-70b-versatile"

# ── Detection Settings ────────────────────────────────────────
KEYWORD_ONLY_MODE    = False   # Use both layers (recommended)
CONFIDENCE_THRESHOLD = 0.5     # Min LLM confidence to trust result
MAX_INPUT_CHARS      = 800     # Max chars sent to LLM per article
API_CALL_DELAY       = 0.5     # Groq is fast; shorter delay than HF

# ── Adverse Category Taxonomy (unchanged) ─────────────────────
ADVERSE_CATEGORIES = [
    "Fraud & Financial Crime",
    "Legal & Regulatory",
    "Corruption & Bribery",
    "ESG & Environmental",
    "Cybersecurity & Data",
    "Labor & Human Rights",
    "Reputational",
    "Operational Risk",
    "Sanctions & Watchlist",
    "Not Adverse",
]

print(f"✅ Configuration updated.")
print(f"   Entity    : {entity_name}")
print(f"   LLM       : Groq / {GROQ_MODEL}")


✅ Configuration updated.
   Entity    : Tesla
   LLM       : Groq / llama-3.3-70b-versatile


In [47]:

def load_phase1_data(csv_path: str) -> pd.DataFrame:
    """
    Loads Phase 1 news data.
    Priority: in-memory df_news (same session) → CSV fallback.
    """
    try:
        ipython = get_ipython()
        if 'df_news' in ipython.user_ns:
            df = ipython.user_ns['df_news']
            if isinstance(df, pd.DataFrame) and not df.empty:
                print(f"✅ Loaded df_news from session memory ({len(df)} articles).")
                return df.copy()
    except Exception:
        pass

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"✅ Loaded from CSV: '{csv_path}' ({len(df)} articles).")
        return df

    raise FileNotFoundError(
        f"Phase 1 data not found in memory or at '{csv_path}'.\n"
        f"Please run Phase 1 first."
    )

KEYWORD_TAXONOMY = {
    "Fraud & Financial Crime": [
        r"\bfraud\b", r"\bembezzl", r"\bponzi\b", r"\bmoney launder",
        r"\baccounting fraud\b", r"\bfinancial crime\b", r"\bwire fraud\b",
        r"\binsider trading\b", r"\bsecurities fraud\b", r"\bpyramid scheme\b",
        r"\bforged\b", r"\bmisappropriat", r"\bdeceptive practice",
    ],
    "Legal & Regulatory": [
        r"\blawsuit\b", r"\blitigation\b", r"\bsued\b", r"\bindictment\b",
        r"\bclass action\b", r"\bsec investigation\b", r"\bregulatory fine\b",
        r"\bpenalt", r"\bviolation\b", r"\bcourt order\b", r"\bjudgment\b",
        r"\bprosecuted\b", r"\bcriminal charge\b", r"\bdoj\b", r"\bfbi probe\b",
        r"\bsanctioned\b", r"\bsebi\b", r"\bfca investigat",
    ],
    "Corruption & Bribery": [
        r"\bbribery\b", r"\bcorruption\b", r"\bkickback\b", r"\bextortion\b",
        r"\bpolitical scandal\b", r"\bfcpa\b", r"\buk bribery act\b",
        r"\bpayoff\b", r"\billicit payment", r"\bgraft\b",
    ],
    "ESG & Environmental": [
        r"\bpollution\b", r"\benvironmental violation\b", r"\bgreenwash",
        r"\bcarbon emission\b", r"\bclimate fraud\b", r"\bhazardous waste\b",
        r"\boil spill\b", r"\bepa fine\b", r"\benvironmental fine\b",
        r"\bsustainability fraud\b", r"\bdeforestation\b",
    ],
    "Cybersecurity & Data": [
        r"\bdata breach\b", r"\bhacked\b", r"\bcyberattack\b", r"\bransom",
        r"\bprivacy violation\b", r"\bdata leak\b", r"\bpersonal data exposed\b",
        r"\bgdpr fine\b", r"\bcybersecurity incident\b", r"\bidentity theft\b",
        r"\bmalware\b", r"\bphishing\b",
    ],
    "Labor & Human Rights": [
        r"\bchild labor\b", r"\bslave labor\b", r"\bhuman trafficking\b",
        r"\bworker abuse\b", r"\bdiscrimination\b", r"\bharassment\b",
        r"\bunfair wage\b", r"\bwhistleblower\b", r"\bunion busting\b",
        r"\bforced labor\b", r"\bsweatshop\b", r"\bretaliation\b",
    ],
    "Reputational": [
        r"\bscandal\b", r"\bcontroversy\b", r"\bmisconduct\b", r"\bresigned amid\b",
        r"\bforced out\b", r"\bboycott\b", r"\bbacklash\b",
        r"\bpublic outrage\b", r"\baccused of\b", r"\ballegation\b",
    ],
    "Operational Risk": [
        # ── TIGHTENED from original Phase 2 ──
        # Each pattern now requires a harm-context word near the trigger.
        # This eliminates false positives on positive/neutral mentions.

        # Self-driving: only flag when paired with harm words
        r"\bself-driving.{0,40}(crash|accident|death|injur|fail|recall|ban|probe|halt)\b",
        r"(crash|accident|death|injur|fail|recall|ban|probe).{0,40}\bself-driving\b",

        # Autopilot: only flag when paired with failure/accident words
        r"\bautopilot.{0,40}(crash|malfunction|disengag|death|injur|fail|recall)\b",
        r"(crash|malfunction|death|injur|fail).{0,40}\bautopilot\b",

        # Robotaxi: only flag when paired with service failure or incident words
        r"\brobotaxi.{0,40}(stall|crash|fail|injur|halt|ban|probe|incident|sue|block)\b",
        r"(stall|crash|fail|injur|halt|ban|incident).{0,40}\brobotaxi\b",

        # Unambiguous operational failure terms (no proximity required)
        r"\bproduct recall\b", r"\bsafety failure\b",
        r"\bfatal (crash|accident|collision)\b",
        r"\b(fire|explosion|battery fire)\b.{0,30}\btesla\b",
        r"\btesla.{0,30}\b(fire|explosion|battery fire)\b",
        r"\bproduction halt\b", r"\bmanufacturing defect\b",
    ],
    "Sanctions & Watchlist": [
        r"\bofac\b", r"\bsanctions list\b", r"\bblacklist\b", r"\bwatchlist\b",
        r"\bun sanctions\b", r"\beu sanctions\b", r"\bterrorist financ",
        r"\bproliferation\b", r"\bdebarred\b", r"\bdesignated entity\b",
    ],
}


# Positive override patterns: if ANY of these match the title,
# the keyword layer returns None regardless of keyword hits.
# This forces the LLM to make a context-aware decision.
#
# Why title only (not summary)?
# Titles carry the editorial framing. A title saying "gets approval"
# is definitively positive. Summaries may contain both positive and
# negative content, making them unreliable for negation detection.
POSITIVE_OVERRIDE_PATTERNS = [
    r"\bgets? (approval|go-ahead|clearance|approved|greenlit)\b",
    r"\b(approved|cleared) (to sell|for sale|by regulator|in [a-z]+)\b",
    r"\b(solves?|solved|breakthrough|bull|bullish|record (profit|revenue|delivery|sales))\b",
    r"\b(launches?|unveils?|releases?|expands?|wins? (contract|deal|bid))\b",
    r"\b(raises? (funding|capital)|ipo|partnership|collaboration)\b",
    r"\befficiency (champ|leader|record)\b",
]


def keyword_screen(title: str, summary: str) -> dict | None:
    """
    Layer 1: Fast keyword-based adverse detection.

    PATCH CHANGES vs original:
      1. Negation guard checks the title FIRST.
         If title has positive framing → returns None immediately,
         bypassing keyword scan and sending article to LLM.
      2. Keyword patterns for Operational Risk are now context-aware
         (harm words required in proximity).

    Returns:
        dict  → keyword match found, article is adverse
        None  → no match, defer to LLM layer
    """
    title_lower = title.lower()
    text        = f"{title} {summary}".lower()

    # ── Negation Guard: check title for positive framing ──────
    # Applied to title only (more reliable signal than summary)
    for pat in POSITIVE_OVERRIDE_PATTERNS:
        if re.search(pat, title_lower):
            # Positive framing detected — skip keyword layer entirely
            # LLM will decide with full context awareness
            return None

    # ── Keyword Scan ──────────────────────────────────────────
    matched_categories = []
    matched_keywords   = []

    for category, patterns in KEYWORD_TAXONOMY.items():
        for pattern in patterns:
            match = re.search(pattern, text)
            if match:
                if category not in matched_categories:
                    matched_categories.append(category)
                kw = match.group().strip()
                if kw not in matched_keywords:
                    matched_keywords.append(kw)

    if matched_categories:
        primary_category = matched_categories[0]
        all_cats = ", ".join(matched_categories)
        kw_str   = ", ".join(matched_keywords[:5])

        return {
            "is_adverse"       : True,
            "adverse_category" : primary_category,
            "confidence_score" : 0.85,
            "reason"           : f"Keyword match(es): [{kw_str}]. Categories detected: {all_cats}.",
            "detection_layer"  : "keyword",
        }

    return None  # No match → LLM decides


groq_client = Groq(api_key=GROQ_API_KEY)


def build_classification_prompt(entity: str, title: str, summary: str) -> str:

    combined = f"{title}. {summary}"

    if len(combined) > MAX_INPUT_CHARS:
        combined = combined[:MAX_INPUT_CHARS] + "..."

    categories_str = "\n".join(f"  - {c}" for c in ADVERSE_CATEGORIES)

    system_prompt = """
You are a senior compliance analyst specializing in adverse media screening.

Your job is to identify MATERIAL adverse media only.

You respond ONLY with valid JSON.
No markdown.
No explanations outside JSON.
"""

    user_prompt = f"""
Classify the following news article about "{entity}".

Article:
{combined}

Adverse categories:
{categories_str}

Important Classification Rules:

- Operational Risk = product failures, crashes, safety incidents, recalls, malfunctions, service outages.
  Examples:
  - autopilot crash
  - robotaxi stalling
  - battery fire
  - recall

- Reputational Risk = scandals, executive misconduct, corruption allegations,
  public controversies, boycotts, major public backlash.

- Regulatory Risk = investigations, fines, sanctions, regulatory enforcement actions.

- Legal Risk = lawsuits, litigation, criminal proceedings.

- Financial Risk = bankruptcy, insolvency, fraud, accounting irregularities.

DO NOT classify as adverse merely because:

- a competitor performs better
- analyst criticism
- stock volatility
- valuation concerns
- opinion/editorial commentary
- product comparisons
- market competition
- speculative future risks

Examples of NOT ADVERSE:

- BYD vs Tesla comparisons
- Tesla vs Rivian reliability comparisons
- Regulatory approvals
- Product launches
- Positive reviews
- Industry analysis
- Parking tickets
- Minor customer complaints

Only classify as adverse when:

1. The article describes a material negative event.
2. "{entity}" is the subject of that event.
3. The event creates genuine operational, legal, financial,
   regulatory, or reputational risk.

Return JSON with exactly:

{{
    "is_adverse": true/false,
    "adverse_category": "<category>",
    "confidence_score": 0.0-1.0,
    "reason": "<brief explanation>"
}}

Respond with JSON only.
"""

    return system_prompt, user_prompt


def parse_llm_response(raw_text: str) -> dict:
    """
    Extracts and validates the JSON object from LLM output.
    Identical to original Phase 2 — contract is unchanged.
    """
    fallback = {
        "is_adverse"       : False,
        "adverse_category" : "Not Adverse",
        "confidence_score" : 0.0,
        "reason"           : "LLM response could not be parsed.",
        "detection_layer"  : "llm_parse_error",
    }

    if not raw_text:
        return fallback

    # Strip markdown fences if model added them despite instructions
    raw_text = re.sub(r'```json|```', '', raw_text).strip()

    # Extract first { ... } block
    json_match = re.search(r'\{[^{}]*\}', raw_text, re.DOTALL)
    if not json_match:
        return fallback

    try:
        parsed = json.loads(json_match.group())

        # Ensure all required keys exist
        for key in ["is_adverse", "adverse_category", "confidence_score", "reason"]:
            if key not in parsed:
                parsed[key] = fallback[key]

        # Normalize and clamp types
        parsed["is_adverse"]       = bool(parsed["is_adverse"])
        parsed["confidence_score"] = float(max(0.0, min(1.0, parsed.get("confidence_score", 0.0))))
        parsed["adverse_category"] = str(parsed.get("adverse_category", "Not Adverse"))
        parsed["reason"]           = str(parsed.get("reason", ""))
        parsed["detection_layer"]  = "llm"

        # Validate category is in taxonomy; correct if hallucinated
        if parsed["adverse_category"] not in ADVERSE_CATEGORIES:
            parsed["adverse_category"] = "Reputational"

        return parsed

    except (json.JSONDecodeError, ValueError):
        return fallback


@retry(
    retry=retry_if_exception_type(RateLimitError),
    wait=wait_exponential(multiplier=2, min=5, max=30),
    stop=stop_after_attempt(3)
)
def call_groq_api(system_prompt: str, user_prompt: str) -> str:
    """
    Calls the Groq Chat Completions API for a single article.

    Why chat completions (not text-generation):
    - Groq's SDK uses OpenAI-compatible chat format
    - System message lets us enforce the compliance analyst persona
    - More reliable JSON output than pure text-completion models

    Parameters:
      temperature=0.0  → fully deterministic, no randomness
                          Critical for compliance use cases where
                          the same article must always get the same result.
      max_tokens=250   → JSON response is short; 250 is generous.

    Retries automatically on RateLimitError (tenacity decorator).

    Returns:
        Raw string content from the model.
    """
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.0,    # Deterministic output for compliance consistency
        max_tokens=250,     # JSON response is compact; 250 tokens is sufficient
    )
    return response.choices[0].message.content.strip()


def classify_article_with_llm(entity: str, title: str, summary: str) -> dict:
    """
    Layer 2: Full LLM classification using Groq.

    Builds prompt → calls Groq → parses JSON response.
    All exceptions are caught so one failure doesn't halt the batch.

    Note: No api_token argument needed — Groq client is initialized
    globally from GROQ_API_KEY in config.
    """
    try:
        system_prompt, user_prompt = build_classification_prompt(entity, title, summary)
        raw_text = call_groq_api(system_prompt, user_prompt)
        result   = parse_llm_response(raw_text)
        return result

    except RateLimitError:
        return {
            "is_adverse"       : False,
            "adverse_category" : "Not Adverse",
            "confidence_score" : 0.0,
            "reason"           : "Groq rate limit exceeded after retries. Consider switching to llama-3.1-8b-instant.",
            "detection_layer"  : "llm_rate_limit",
        }
    except APIStatusError as e:
        return {
            "is_adverse"       : False,
            "adverse_category" : "Not Adverse",
            "confidence_score" : 0.0,
            "reason"           : f"Groq API error: {e.status_code} — {str(e.message)}",
            "detection_layer"  : "llm_api_error",
        }
    except Exception as e:
        return {
            "is_adverse"       : False,
            "adverse_category" : "Not Adverse",
            "confidence_score" : 0.0,
            "reason"           : f"Unexpected LLM error: {str(e)}",
            "detection_layer"  : "llm_error",
        }



def run_adverse_detection(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    """
    Master orchestrator: two-layer detection across all articles.

    Layer 1 (Keyword): Fast pattern match. Returns result if confident.
    Layer 2 (Groq LLM): Deep classification for ambiguous articles.

    Hybrid merge logic:
      - Keyword hit + LLM agrees (conf >= threshold)  → use LLM result (richer reason)
      - Keyword hit + LLM disagrees (conf < threshold) → trust keyword result
      - No keyword hit                                 → use LLM result exclusively
      - KEYWORD_ONLY_MODE = True                       → skip LLM entirely

    Output columns added to df:
      is_adverse, adverse_category, confidence_score, reason, detection_layer
    """
    results  = []
    llm_calls = 0
    kw_hits   = 0

    print(f"\n🔍 Running adverse detection on {len(df)} articles...")
    print(f"   LLM Backend : Groq / {GROQ_MODEL}")
    print(f"   Mode        : {'Keyword only' if KEYWORD_ONLY_MODE else 'Keyword + LLM (hybrid)'}")
    print("-" * 60)

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Screening articles"):
        title   = str(row.get('title', ''))
        summary = str(row.get('article_summary', ''))

        # ── Layer 1: Keyword Screen ──────────────────────────
        kw_result = keyword_screen(title, summary)

        if KEYWORD_ONLY_MODE:
            # Keyword-only mode: use keyword result or mark not adverse
            if kw_result:
                results.append(kw_result)
                kw_hits += 1
            else:
                results.append({
                    "is_adverse"       : False,
                    "adverse_category" : "Not Adverse",
                    "confidence_score" : 0.9,
                    "reason"           : "No adverse keywords detected.",
                    "detection_layer"  : "keyword",
                })
            continue

        # ── Layer 2: Groq LLM ────────────────────────────────
        if kw_result:
            kw_hits += 1
            # Keyword matched — run LLM to refine reason and category
            llm_result = classify_article_with_llm(entity, title, summary)
            llm_calls += 1

            if llm_result["confidence_score"] >= CONFIDENCE_THRESHOLD:
                # LLM is confident — use its richer reasoning
                # Take the higher confidence of the two
                llm_result["confidence_score"] = max(
                    llm_result["confidence_score"],
                    kw_result["confidence_score"]
                )
                results.append(llm_result)
            else:
                # LLM uncertain — trust the keyword signal
                results.append(kw_result)
        else:
            # No keyword match — LLM decides entirely
            llm_result = classify_article_with_llm(entity, title, summary)
            results.append(llm_result)
            llm_calls += 1

        time.sleep(API_CALL_DELAY)  # Respect Groq rate limits

    print("-" * 60)
    print(f"✅ Detection complete.")
    print(f"   Keyword layer hits : {kw_hits}")
    print(f"   Groq API calls     : {llm_calls}")

    # Build enriched DataFrame
    df_out     = df.copy().reset_index(drop=True)
    results_df = pd.DataFrame(results)

    df_out["is_adverse"]       = results_df["is_adverse"].values
    df_out["adverse_category"] = results_df["adverse_category"].values
    df_out["confidence_score"] = results_df["confidence_score"].values
    df_out["reason"]           = results_df["reason"].values
    df_out["detection_layer"]  = results_df["detection_layer"].values

    # Sort: adverse first, then by confidence descending
    df_out = df_out.sort_values(
        by=["is_adverse", "confidence_score"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return df_out


print("✅ All patched functions defined (Groq + tightened keywords).")

✅ All patched functions defined (Groq + tightened keywords).


In [48]:
df_news     = load_phase1_data(PHASE1_CSV_PATH)
df_analyzed = run_adverse_detection(df_news, entity_name)

df_analyzed.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\n💾 Updated results saved to: '{OUTPUT_CSV_PATH}'")
print("   → Phase 3 (Risk Scoring) loads this with: pd.read_csv('" + OUTPUT_CSV_PATH + "')")

✅ Loaded df_news from session memory (30 articles).

🔍 Running adverse detection on 30 articles...
   LLM Backend : Groq / llama-3.3-70b-versatile
   Mode        : Keyword + LLM (hybrid)
------------------------------------------------------------


Screening articles:   0%|          | 0/30 [00:00<?, ?it/s]

------------------------------------------------------------
✅ Detection complete.
   Keyword layer hits : 3
   Groq API calls     : 30

💾 Updated results saved to: 'adverse_detection_Tesla_phase2.csv'
   → Phase 3 (Risk Scoring) loads this with: pd.read_csv('adverse_detection_Tesla_phase2.csv')


In [49]:
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

adverse_df     = df_analyzed[df_analyzed['is_adverse'] == True]
non_adverse_df = df_analyzed[df_analyzed['is_adverse'] == False]

print("=" * 70)
print(f"🚨 ADVERSE DETECTION SUMMARY (PATCHED) — Entity: '{entity_name}'")
print("=" * 70)
print(f"  Total articles screened   : {len(df_analyzed)}")
print(f"  ⚠️  Adverse articles       : {len(adverse_df)} "
      f"({100*len(adverse_df)/max(len(df_analyzed),1):.1f}%)")
print(f"  ✅ Non-adverse articles    : {len(non_adverse_df)}")

if not adverse_df.empty:
    print(f"  📊 Avg confidence (adverse): {adverse_df['confidence_score'].mean():.2f}")

# --- Risk Category Breakdown ---
print("\n📌 Adverse Articles by Risk Category:")
print("-" * 50)
if not adverse_df.empty:
    for cat, count in adverse_df['adverse_category'].value_counts().items():
        print(f"  {cat:<35} {count:>3}  {'█' * count}")
else:
    print("  No adverse articles found.")

# --- Detection Layer Breakdown ---
print("\n🔬 Detection Layer Used:")
print("-" * 50)
for layer, count in df_analyzed['detection_layer'].value_counts().items():
    print(f"  {layer:<35} {count}")

# --- False Positive Audit ---
# Shows articles the patched keyword layer correctly let through to LLM
# (negation guard fired)
print("\n🔎 Articles Routed via Negation Guard (positive framing detected in title):")
print("-" * 70)
# These will show detection_layer = 'llm' with is_adverse = False
fp_candidates = df_analyzed[
    (df_analyzed['detection_layer'] == 'llm') &
    (df_analyzed['is_adverse'] == False)
][['title', 'adverse_category', 'confidence_score', 'reason']].head(5)
if not fp_candidates.empty:
    display(fp_candidates)
else:
    print("  None.")

# --- Full DataFrame ---
print("\n📋 Full Analyzed DataFrame (adverse first):")
display_cols = ["title", "source", "publication_date",
                "is_adverse", "adverse_category", "confidence_score",
                "reason", "detection_layer"]
display(df_analyzed[display_cols])

# --- Top Adverse Hits ---
if not adverse_df.empty:
    print("\n🚨 Top Adverse Articles (highest confidence):")
    print("=" * 70)
    for i, (_, row) in enumerate(adverse_df.head(5).iterrows(), 1):
        print(f"\n  [{i}] {row['title']}")
        print(f"      Source     : {row['source']}")
        print(f"      Date       : {row['publication_date']}")
        print(f"      Category   : {row['adverse_category']}")
        print(f"      Confidence : {row['confidence_score']:.2f}")
        print(f"      Layer      : {row['detection_layer']}")
        print(f"      Reason     : {row['reason']}")

print("\n" + "=" * 70)
print("✅ Phase 2 (patched) complete. df_analyzed ready for Phase 3.")
print(f"   Adverse : {len(adverse_df)} | Non-adverse : {len(non_adverse_df)} | Total : {len(df_analyzed)}")
print("=" * 70)

🚨 ADVERSE DETECTION SUMMARY (PATCHED) — Entity: 'Tesla'
  Total articles screened   : 30
  ⚠️  Adverse articles       : 4 (13.3%)
  ✅ Non-adverse articles    : 26
  📊 Avg confidence (adverse): 0.88

📌 Adverse Articles by Risk Category:
--------------------------------------------------
  Operational Risk                      4  ████

🔬 Detection Layer Used:
--------------------------------------------------
  llm                                 30

🔎 Articles Routed via Negation Guard (positive framing detected in title):
----------------------------------------------------------------------


,title,adverse_category,confidence_score,reason
4,TSLA Stock Sinks As SpaceX IPO Nears — Analyst Says Investors Are Selling Te...,Not Adverse,1.0,"Article discusses stock volatility and analyst speculation, which is not a m..."
5,Prediction: Here's When SpaceX Will Merge With Tesla - Yahoo Finance,Not Adverse,1.0,Article is speculative about a potential merger and does not describe a mate...
6,"Gift Nifty Signals Weak Open Amid Crude, US-Iran Tensions",Not Adverse,1.0,Article does not mention Tesla as subject of a material negative event
7,Cathie Wood Asks Elon Musk How Tesla Handles 'Parking And Traffic Violations...,Not Adverse,1.0,Parking ticket is a minor incident and not a material negative event
8,How Tesla’s Stock Listing in 2010 Enabled SpaceX’s I.P.O. - The New York Times,Not Adverse,1.0,"Article discusses stock performance and IPO plans, no material negative event"



📋 Full Analyzed DataFrame (adverse first):


,title,source,publication_date,is_adverse,adverse_category,confidence_score,reason,detection_layer
0,Tesla’s Robotaxi Falls Short With Long Waits and Stalled Rides - Bloomberg.com,Bloomberg.com,2026-06-10,True,Operational Risk,0.9,"Tesla's Robotaxi experienced product malfunctions, including long waits and ...",llm
1,Tesla robotaxis stall as Musk’s self-driving hype hits real-world traffic - ...,Los Angeles Times,2026-06-10,True,Operational Risk,0.9,Tesla robotaxis stalling,llm
2,Tesla crashes into Redmond garage after reported autopilot malfunction - KIN...,KING5.com,2026-06-08,True,Operational Risk,0.9,Tesla autopilot malfunction resulting in a crash,llm
3,O.C. Uber customer says driver asleep in Tesla on 405 Freeway - KTLA,KTLA,2026-06-10,True,Operational Risk,0.8,Tesla vehicle involved in safety incident with driver asleep,llm
4,TSLA Stock Sinks As SpaceX IPO Nears — Analyst Says Investors Are Selling Te...,newsable_asianetnews,2026-06-11,False,Not Adverse,1.0,"Article discusses stock volatility and analyst speculation, which is not a m...",llm
5,Prediction: Here's When SpaceX Will Merge With Tesla - Yahoo Finance,Yahoo Finance,2026-06-11,False,Not Adverse,1.0,Article is speculative about a potential merger and does not describe a mate...,llm
6,"Gift Nifty Signals Weak Open Amid Crude, US-Iran Tensions",fivepaisa,2026-06-11,False,Not Adverse,1.0,Article does not mention Tesla as subject of a material negative event,llm
7,Cathie Wood Asks Elon Musk How Tesla Handles 'Parking And Traffic Violations...,Yahoo Finance,2026-06-11,False,Not Adverse,1.0,Parking ticket is a minor incident and not a material negative event,llm
8,How Tesla’s Stock Listing in 2010 Enabled SpaceX’s I.P.O. - The New York Times,The New York Times,2026-06-11,False,Not Adverse,1.0,"Article discusses stock performance and IPO plans, no material negative event",llm
9,What Does Denmark's FSD Approval Mean for Tesla in Europe? - Yahoo Finance S...,Yahoo Finance Singapore,2026-06-11,False,Not Adverse,1.0,"Article discusses regulatory approval, which is not adverse",llm



🚨 Top Adverse Articles (highest confidence):

  [1] Tesla’s Robotaxi Falls Short With Long Waits and Stalled Rides - Bloomberg.com
      Source     : Bloomberg.com
      Date       : 2026-06-10
      Category   : Operational Risk
      Confidence : 0.90
      Layer      : llm
      Reason     : Tesla's Robotaxi experienced product malfunctions, including long waits and stalled rides

  [2] Tesla robotaxis stall as Musk’s self-driving hype hits real-world traffic - Los Angeles Times
      Source     : Los Angeles Times
      Date       : 2026-06-10
      Category   : Operational Risk
      Confidence : 0.90
      Layer      : llm
      Reason     : Tesla robotaxis stalling

  [3] Tesla crashes into Redmond garage after reported autopilot malfunction - KING5.com
      Source     : KING5.com
      Date       : 2026-06-08
      Category   : Operational Risk
      Confidence : 0.90
      Layer      : llm
      Reason     : Tesla autopilot malfunction resulting in a crash

  [4] O.C. Uber c